In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install torch
!pip install scikit-learn

In [3]:
import torch

In [4]:
# To check if cuda gpus available
torch.cuda.is_available()
# To check number of gpus
torch.cuda.device_count()
# To get the gpu name
torch.cuda.get_device_name(0)
# To get the actual device
torch.cuda.device(0)

### TENSORS

In [5]:
# Tensor is a high dimensional array

# In numpy we do the below
import numpy as np

arr = np.array([[1,2,3], [4,5,6]])
arr

array([[1, 2, 3],
       [4, 5, 6]])

In [6]:
# In torch we do. Notice how the outputs are in floats
tensor = torch.Tensor([[1,2,3], [4,5,6]])
tensor

tensor([[1., 2., 3.],
        [4., 5., 6.]])

In [7]:
arr * 5

array([[ 5, 10, 15],
       [20, 25, 30]])

In [8]:
tensor * 5

tensor([[ 5., 10., 15.],
        [20., 25., 30.]])

In [9]:
arr.sum()

np.int64(21)

In [10]:
tensor.sum()

tensor(21.)

In [11]:
# Create tensors from numpy arrays
tensor = torch.from_numpy(arr)
tensor

tensor([[1, 2, 3],
        [4, 5, 6]])

In [12]:
np.ones((2,4))

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [13]:
torch.ones((2,4))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [14]:
np.random.random((2,4))

array([[0.25466107, 0.26409934, 0.51216283, 0.89905448],
       [0.99381183, 0.29293965, 0.12068612, 0.8496378 ]])

In [15]:
torch.rand((2,4))

tensor([[0.6329, 0.0562, 0.3003, 0.9175],
        [0.2843, 0.5532, 0.0168, 0.0022]])

In [16]:
arr.shape # shape of array 
arr.dtype # datatype of array
arr.device # which device array is on

'cpu'

In [17]:
tensor.shape
tensor.dtype
tensor.device

device(type='cpu')

In [18]:
# Numpy doesnt support gpu parallelism. Only torch does

tensor = tensor.to('cuda')

In [19]:
tensor.sum()

tensor(21, device='cuda:0')

In [20]:
# to bring back tensor from gpu to cpu
# needed if we want to convert it back to numpy array

tensor.cpu().numpy()

array([[1, 2, 3],
       [4, 5, 6]])

In [21]:
# Best practice is
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# note if you have a tensor on one device and a model on another they will not work together

### Pytorch and its support

Pytorch supports automatic differentiation which is needed for optimization or backpropagation in neural networks

In [22]:
import torch

In [23]:
a = torch.tensor([2., 3.], requires_grad = True)
b = torch.tensor([6., 4.], requires_grad = True)

f = 3 * a **3 - b **2

In [24]:
# Using automatic differentiation engine in use
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [25]:
# Backpropgating the gradient. The gradient we pass tells us how import each tensor is. Here a & b equally imp so [1,1]

f.backward(gradient = torch.tensor([1,1]))

In [26]:
print(a.grad) # Derivative of f wrt a is 9a**2 => 36, 81

tensor([36., 81.])


In [27]:
print(b.grad)# Derivative of f wrt b is -2b => -12, -8

tensor([-12.,  -8.])


# NEURAL NETWORKS

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# nn is used for all the layers and all neural networks building
# functional is used for activation functions- functions applied after layers to break linearity
# optim is for optimizers
# utils needed to load data

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [29]:
# Getting dataset from scikit learn- make sure it fits into pytorch - transform it into a tensor dataset

# Means we get numpy arrays X and y: X contains info about a tumour and y is going to be 0 (malignant) or 1 (benign)
X, y = load_breast_cancer(return_X_y = True)

# Split into training and testing set to prevent biad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

# Since neural networks are scale sensitive we need to scale the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # we want to use to params learned from scaling training dataset to scale test dataset 

In [30]:
X_train_scaled

array([[-0.60063367, -1.0352601 , -0.58964387, ..., -0.98967266,
         0.35791171, -0.45345716],
       [-1.55840815,  0.41231353, -1.52390859, ..., -1.29094538,
         0.11523591, -0.38662001],
       [ 1.36234684,  0.65036966,  1.33393971, ...,  0.87646034,
         0.17111521, -0.056817  ],
       ...,
       [-0.34061733, -1.42961571, -0.35825088, ...,  0.03841296,
        -0.82672934,  0.07411808],
       [ 0.56229656,  0.9557548 ,  0.46466372, ..., -0.20027142,
         0.07691868,  0.01878568],
       [-0.48348345, -0.17441066, -0.51596238, ..., -0.43259089,
         0.13279798, -0.6753346 ]])

In [31]:
y_train

array([1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0,
       0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1,
       1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0,
       0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1,

In [32]:
# Take all the above - turn to tensors - then to a tensor dataset- then use the dataloader to load the data for training process
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1) # to add extra dimensions for training
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1) # to add extra dimensions for training


In [33]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [34]:
# to determine batch size. Here 455- so 32 is good match
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [35]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

In [36]:
# Define our neural network
class BCNet(nn.Module): # Initialize breast cancer net

    def __init__(self):
        super(BCNet, self).__init__()

        # Create fully connected layer- dense layer. Input is 30 because [455, 30] means we have 30 features per instance
        self.fc1 = nn.Linear(30, 64) # output neurons we take 64
        self.fc2 = nn.Linear(64, 32) # input is output of previous layer
        self.fc3 = nn.Linear(32, 1) # input is output of previous layer and produces actual o/p 1 value

        self.dropout = nn.Dropout(0.5)

    # Passing data through this neural network
    def forward(self, x):
        # take input or batch, feed it thru n/w and get result of taking input & multiplying with wts of fcl & adding bias to get o/p. 
        x = F.relu(self.fc1(x)) # to break linearity we apply a non linear activation function
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x)) # sigmoid produces value b/w 0 and 1- which we need to produce probability

        return x

In [37]:
model = BCNet() # create model

In [38]:
# define 2 things: criterion (loss function) and optimizer
criterion = nn.BCELoss() # if you have yes or no question we use this
optimizer = optim.Adam(model.parameters(), lr = 0.001) # optimizer optimizes params of the model

In [39]:
# define the training loop
epochs = 20 # do 20 iterations on same data

for epoch in range(epochs):
    model.train()
    running_loss = 0.0 # starts at 0.0

    # iterate over data loader to get individual batches
    for x_batch, y_batch in train_loader:
        # feed data thru model, get predictions which is initially random then take ground truths and calculate loss or how wrong model is
        optimizer.zero_grad() # reset or remove all gradients

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        # then we take loss which is a function and backpropagate it (take derivative of the function) and try to find min in that function
        loss.backward()
        # take a step with the optimizer in that direction depending on learning rate (large/small step)
        optimizer.step() 

        running_loss += loss.item() 

    print(f'Epoch {epoch+1}: Loss was {running_loss/ len(train_loader)}')
    

Epoch 1: Loss was 0.6031570434570312
Epoch 2: Loss was 0.4776306887467702
Epoch 3: Loss was 0.3333691696325938
Epoch 4: Loss was 0.22951993495225906
Epoch 5: Loss was 0.18238257616758347
Epoch 6: Loss was 0.12926549961169562
Epoch 7: Loss was 0.10654382258653641
Epoch 8: Loss was 0.0821689469118913
Epoch 9: Loss was 0.07036561531325182
Epoch 10: Loss was 0.07584070911010106
Epoch 11: Loss was 0.08057290005187194
Epoch 12: Loss was 0.06892369429891308
Epoch 13: Loss was 0.06064511189858119
Epoch 14: Loss was 0.0642524761458238
Epoch 15: Loss was 0.05359840430319309
Epoch 16: Loss was 0.05347386443366607
Epoch 17: Loss was 0.052414193252722424
Epoch 18: Loss was 0.052474667131900785
Epoch 19: Loss was 0.04496894267698129
Epoch 20: Loss was 0.04067989050721129


In [40]:
# Define evaluation logic. Evaluate model on unseen data
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [41]:
accuracy

0.9649122953414917